# 숫자·영어·한글 번역 - 데이터 업로드부터 학습하기

**원본 데이터 업로드 → 데이터 분할 → 모델 3개 학습 → 테스트 정확도 측정 → 내 입력 번역 → 모델 다운로드** 순서로 실행합니다.
기존 `.pt` 파일이나 외부 JSON 파일은 필요하지 않습니다.

1. 제출 ZIP을 PC에서 압축 해제하고 이 노트북을 Google Colab에서 엽니다.
2. 가능하면 코랩의 런타임 유형을 GPU로 선택합니다. CPU에서도 실행할 수 있습니다.
3. 위에서부터 실행하고, 업로드 창에서 `num2txt.txt`, `num2kor.txt` 두 파일을 함께 선택합니다.
4. 학습과 테스트가 끝나면 마지막 번역 셀에서 원하는 입력을 시험합니다.
5. 마지막 다운로드 셀을 실행하면 이번에 학습한 모델을 ZIP으로 받습니다.

기본값은 전체 데이터, 모델당 최대 20 epoch입니다. `QUICK_RUN=True`는 소량 실행 확인용입니다.
이 노트북은 **새 가중치에서 처음부터 학습**합니다. 이전 추가 학습 모델의 정확도를 그대로 재현하거나 표시하지 않으며,
이번 실행의 테스트 결과를 직접 계산합니다. ZIP에 포함된 원본 데이터 대신 같은 형식의 두 파일을 올려도 됩니다.

## 1. 실행 환경과 설정
숫자↔영어, 숫자↔한글, 영어↔한글 모델을 각각 학습합니다.
단층 단방향 GRU Encoder와 Decoder에 Dot-product Attention을 연결합니다.
Embedding 64, hidden 128, 배치 128, Adam 학습률 0.001을 사용합니다.

In [1]:
from pathlib import Path
import random
import re
import time

import importlib.util
import subprocess
import sys
import zipfile
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch>=2.6,<3"])

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

SEED = 42
QUICK_RUN = False #@param {type:"boolean"}
DATA_DIR = Path.cwd() / "uploaded_data"
MODEL_DIR = Path.cwd()  # 노트북과 같은 폴더의 .pt를 저장하고 로드합니다.
PAIRS = [("num", "en"), ("num", "ko"), ("en", "ko")]
EMBED_SIZE = 64                  # 토큰 하나를 표현하는 벡터 길이
HIDDEN_SIZE = 128                # GRU가 기억하는 벡터 길이
BATCH_SIZE = 128
EPOCHS = 3 if QUICK_RUN else 20
LEARNING_RATE = 0.001
MAX_NEW_TOKENS = 40              # 제공 데이터의 최대 출력 토큰 수보다 크게 설정
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PREFIX = "demo_" if QUICK_RUN else ""

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    torch.set_num_threads(2)
MODEL_DIR.mkdir(exist_ok=True)
print("장치:", DEVICE, "| 빠른 실행:", QUICK_RUN)


장치: cuda | 빠른 실행: False


## 2. 원본 데이터 두 파일 업로드
`num2txt.txt`는 영어, `num2kor.txt`는 한글 표현입니다.
각 줄은 `숫자 문자열` + **탭** + `해당 언어 표현` 형식입니다. 예: `14	fourteen`.
숫자는 문자열로 유지하며, 중복 숫자와 잘못된 행은 오류로 안내합니다.
두 파일의 숫자 목록이 달라도 됩니다. 각 언어쌍에 필요한 표현이 있는 숫자만 사용합니다.
영어↔한글 학습에는 두 파일의 공통 숫자가 필요합니다.

In [2]:
from google.colab import files

uploaded_data = files.upload()
required_files = ("num2txt.txt", "num2kor.txt")
missing_files = [name for name in required_files if name not in uploaded_data]
if missing_files:
    raise ValueError("두 원본 파일을 함께 선택하세요. 빠진 파일: " + ", ".join(missing_files))
DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in required_files:
    (DATA_DIR / name).write_bytes(uploaded_data[name])
print("원본 데이터 업로드 완료:", ", ".join(required_files))


Saving num2kor.txt to num2kor.txt
Saving num2txt.txt to num2txt.txt
원본 데이터 업로드 완료: num2txt.txt, num2kor.txt


## 3. 숫자 기준 결합과 분할
행 순서가 아닌 숫자 문자열로 두 파일을 연결합니다.
합집합을 만든 뒤 숫자 단위로 자릿수별 약 80/10/10으로 나눕니다.
같은 숫자의 정방향과 역방향이 학습과 테스트에 나뉘는 일을 막습니다.

In [3]:
def read_file(path):
    values = {}
    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        parts = line.split("\t")
        if len(parts) != 2 or not parts[0].isascii() or not parts[0].isdigit() or not parts[1]:
            raise ValueError(f"{path.name}의 {line_number}행을 확인하세요.")
        number, text = parts
        if number in values:
            raise ValueError(f"중복 숫자: {number}")
        values[number] = text
    return values


def load_rows(data_dir):
    korean = read_file(data_dir / "num2kor.txt")
    english = read_file(data_dir / "num2txt.txt")
    rows = []
    for number in sorted(korean.keys() | english.keys()):
        row = {"num": number}
        if number in english:
            row["en"] = english[number]
        if number in korean:
            row["ko"] = korean[number]
        rows.append(row)
    print(f"데이터: 영어 {len(english):,}개 | 한글 {len(korean):,}개 | "
          f"공통 숫자 {len(english.keys() & korean.keys()):,}개")
    return rows


def rows_for_pair(rows, pair):
    # 두 표현이 모두 있는 숫자만 해당 언어쌍의 정답 예제로 쓴다.
    return [row for row in rows if all(language in row for language in pair)]


def split_rows(rows):
    # 자릿수별로 섞고 80/10/10으로 나눈다. 모든 모델이 같은 분할을 쓴다.
    rng = random.Random(SEED)
    train, valid, test = [], [], []
    for digits in sorted({len(row["num"]) for row in rows}):
        group = [row for row in rows if len(row["num"]) == digits]
        rng.shuffle(group)
        first, second = int(len(group) * 0.8), int(len(group) * 0.9)
        train.extend(group[:first])
        valid.extend(group[first:second])
        test.extend(group[second:])
    # zero/영은 단독 0에서만 등장하므로 0은 학습에 둔다. 테스트 숫자는 유지한다.
    zero = next((row for row in valid if row["num"] == "0"), None)
    if zero is not None:
        valid.remove(zero)
        train.append(zero)
    return train, valid, test


## 4. 토큰 사전과 배치
숫자·한글은 문자, 영어는 단어 단위로 나눕니다.
사전은 학습 자료로만 만들고 목표 언어 태그를 입력 앞에 붙입니다.
학습 자료의 짧은 숫자는 반복하여 학습 빈도를 높이고, 검증과 테스트는 반복하지 않습니다.

In [4]:
SPECIAL = ["<PAD>", "<BOS>", "<EOS>", "<UNK>", "<num>", "<en>", "<ko>"]
PAD, BOS, EOS, UNK = 0, 1, 2, 3


def tokenize(text, language):
    if not isinstance(text, str):
        raise TypeError('숫자도 "12345"처럼 문자열로 입력하세요.')
    return text.split() if language == "en" else list(text)


def make_vocab(rows):
    tokens = set()
    for row in rows:
        for language in ("num", "en", "ko"):
            if language in row:
                tokens.update(tokenize(row[language], language))
    return SPECIAL + sorted(tokens)


def encode_input(text, source, target, stoi):
    tokens = [f"<{target}>"] + tokenize(text, source) + ["<EOS>"]
    return torch.tensor([stoi.get(t, UNK) for t in tokens], dtype=torch.long)


def encode_target(text, language, stoi):
    tokens = ["<BOS>"] + tokenize(text, language) + ["<EOS>"]
    return torch.tensor([stoi.get(t, UNK) for t in tokens], dtype=torch.long)


def make_examples(rows, pair, stoi):
    a, b = pair
    examples = []
    for row in rows_for_pair(rows, pair):
        # 같은 모델에 두 방향을 모두 학습시킨다.
        for source, target in [(a, b), (b, a)]:
            x = encode_input(row[source], source, target, stoi)
            y = encode_target(row[target], target, stoi)
            examples.append((x, y))
    return examples


def balance_training_rows(rows):
    # 짧은 숫자와 0이 이어지는 숫자의 학습 기회를 늘린다. 검증·테스트에는 쓰지 않는다.
    balanced = []
    for row in rows:
        number = row["num"]
        copies = (256 if number == "0" else 24 if len(number) <= 2 else
                  4 if len(number) == 3 else 2 if "00" in number else 1)
        balanced.extend([row] * copies)
    return balanced


def collate_batch(batch):
    xs, ys = zip(*batch)
    x = pad_sequence(xs, batch_first=True, padding_value=PAD)
    y = pad_sequence(ys, batch_first=True, padding_value=PAD)
    return x, y


def make_loader(rows, pair, stoi, shuffle=False):
    return DataLoader(make_examples(rows, pair, stoi), batch_size=BATCH_SIZE,
                      shuffle=shuffle, collate_fn=collate_batch)


print("숫자 토큰:", tokenize("21546", "num"))
print("영어 토큰:", tokenize("twenty one thousand", "en"))


숫자 토큰: ['2', '1', '5', '4', '6']
영어 토큰: ['twenty', 'one', 'thousand']


## 5. GRU Encoder·Decoder와 Attention
Encoder는 입력 위치별 상태를 만듭니다. Decoder는 Attention으로 입력을 참고하며 다음 토큰을 예측합니다.
PAD는 Attention에서 제외하고 마지막 실제 입력 위치의 상태로 Decoder를 시작합니다.

In [5]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD)
        self.encoder = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.decoder = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size * 2, vocab_size)

    def encode(self, x):
        mask = x != PAD
        encoded, _ = self.encoder(self.embedding(x))
        last_positions = mask.sum(dim=1) - 1
        batch_positions = torch.arange(x.size(0), device=x.device)
        hidden = encoded[batch_positions, last_positions].unsqueeze(0)
        return encoded, hidden, mask

    def decode(self, tokens, hidden, encoded, mask):
        decoded, hidden = self.decoder(self.embedding(tokens), hidden)
        scores = torch.bmm(decoded, encoded.transpose(1, 2))
        scores = scores.masked_fill(~mask.unsqueeze(1), float("-inf"))
        attention = torch.softmax(scores, dim=-1)
        context = torch.bmm(attention, encoded)
        logits = self.output(torch.cat([decoded, context], dim=-1))
        return logits, hidden

    def forward(self, x, target_input):
        encoded, hidden, mask = self.encode(x)
        logits, _ = self.decode(target_input, hidden, encoded, mask)
        return logits


## 6. 학습 함수와 모델 저장
이전 정답으로 다음 토큰을 맞히는 Teacher Forcing과 PAD를 제외한 Cross Entropy를 사용합니다.
검증 손실이 좋아질 때 모델을 저장합니다. 검증 자료가 없으면 정해진 마지막 epoch를 저장합니다.
모델 파일에는 가중치·사전·언어쌍·설정이 함께 들어갑니다. 기록은 메모리에 유지하며 JSON을 만들거나 읽지 않습니다.

In [6]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")
    total_loss, total_tokens = 0.0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x, y[:, :-1])
            answer = y[:, 1:]
            loss_sum = criterion(logits.reshape(-1, logits.size(-1)), answer.reshape(-1))
            token_count = (answer != PAD).sum()
            loss = loss_sum / token_count
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss_sum.item()
            total_tokens += token_count.item()
    return total_loss / total_tokens


def save_model(path, model, vocab, pair, config):
    # 가중치만 있으면 토큰 ID를 복원할 수 없으므로 사전과 설정도 함께 저장한다.
    torch.save({"state_dict": model.state_dict(), "vocab": vocab,
                "pair": list(pair), "config": config}, path)


def load_model(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=True)
    vocab = checkpoint["vocab"]
    config = checkpoint["config"]
    model = Seq2Seq(len(vocab), config["embed_size"], config["hidden_size"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, vocab, tuple(checkpoint["pair"])


def train_pair(pair):
    # 각 모델이 같은 seed에서 시작하도록 설정한다.
    torch.manual_seed(SEED)
    paired_train = rows_for_pair(train_rows, pair)
    paired_valid = rows_for_pair(valid_rows, pair)
    print(f"{pair} 사용 숫자: 학습 {len(paired_train):,}개 | 검증 {len(paired_valid):,}개")
    if not paired_train:
        print(f"건너뜀: {pair}의 두 표현이 모두 있는 학습 숫자가 없습니다.")
        return None
    train_loader = make_loader(balance_training_rows(paired_train), pair, stoi, shuffle=True)
    valid_loader = make_loader(paired_valid, pair, stoi) if paired_valid else None
    if valid_loader is None:
        print("검증 예제가 없어 정해진 마지막 epoch의 모델을 저장합니다.")
    model = Seq2Seq(len(vocab), EMBED_SIZE, HIDDEN_SIZE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    path = MODEL_DIR / f"{PREFIX}{pair[0]}_{pair[1]}.pt"
    config = {"embed_size": EMBED_SIZE, "hidden_size": HIDDEN_SIZE,
              "seed": SEED, "quick_run": QUICK_RUN, "max_new_tokens": MAX_NEW_TOKENS,
              "train_numbers": len(paired_train), "validation_numbers": len(paired_valid),
              "model_selection": "validation_loss" if paired_valid else "last_epoch_without_validation"}
    best_loss = float("inf")
    history = []
    for epoch in range(1, EPOCHS + 1):
        started = time.time()
        train_loss = run_epoch(model, train_loader, optimizer)
        valid_loss = run_epoch(model, valid_loader) if valid_loader is not None else None
        history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss})
        if (valid_loss is not None and valid_loss < best_loss) or (valid_loss is None and epoch == EPOCHS):
            if valid_loss is not None:
                best_loss = valid_loss
            config["best_epoch"] = epoch
            config["valid_loss"] = valid_loss
            save_model(path, model, vocab, pair, config)
        valid_text = f"{valid_loss:.4f}" if valid_loss is not None else "없음"
        print(f"{pair} epoch {epoch}/{EPOCHS} | train {train_loss:.4f} | "
              f"valid {valid_text} | {time.time() - started:.1f}초", flush=True)
    return path


## 7. 번역과 평가 함수
예측은 Greedy 방식으로 출력하고 EOS가 나오면 종료합니다.
정확도는 출력 전체가 정답과 같은 비율(Exact Match)입니다.
숫자 입력의 앞뒤 공백은 제거하며 빈 문자열·문자·쉼표·부호·소수점·중간 공백은 오류로 안내합니다.

In [7]:
def normalize_input(text, language):
    if not isinstance(text, str):
        raise TypeError('숫자도 "12345"처럼 문자열로 입력하세요.')
    if language == "num":
        text = text.strip()
        if not text or not text.isascii() or not text.isdigit():
            raise ValueError('숫자는 0~9로만 입력하세요. 예: "12345". 쉼표·부호·소수점·중간 공백은 사용할 수 없습니다.')
        return text
    if language == "ko":
        text = "".join(text.split())
        # 단위 시작의 일만만 줄인다. 십일만(11만), 백일만(101만)은 보존한다.
        text = re.sub(r"(?<![영일이삼사오육칠팔구십백천])일만", "만", text)
        return re.sub(r"([억만])(?=.)", r"\1 ", text)
    if language == "en":
        text = " ".join(text.lower().split())
        return re.sub(r"\b(hundred|thousand|million|billion)s\b", r"\1", text)
    return text


def output_token_ids(vocab, language):
    # 기존 사전의 숫자·한글·영어 토큰은 문자 종류로 구분할 수 있다.
    def belongs(token):
        if language == "num":
            return len(token) == 1 and token in "0123456789"
        if language == "ko":
            return token == " " or (len(token) == 1 and "가" <= token <= "힣")
        return token.isascii() and token.isalpha()
    return [i for i, token in enumerate(vocab) if i == EOS or belongs(token)]


@torch.no_grad()
def translate(model, vocab, pair, text, source, target, max_new_tokens=MAX_NEW_TOKENS):
    if source == target or {source, target} != set(pair):
        raise ValueError(f"이 모델은 {pair} 사이의 번역만 지원합니다.")
    stoi = {token: i for i, token in enumerate(vocab)}
    text = normalize_input(text, source)
    x = encode_input(text, source, target, stoi).unsqueeze(0).to(DEVICE)
    model.eval()
    encoded, hidden, mask = model.encode(x)
    token = torch.tensor([[BOS]], device=DEVICE)
    result = []
    allowed = torch.zeros(len(vocab), dtype=torch.bool, device=DEVICE)
    allowed[output_token_ids(vocab, target)] = True
    for _ in range(max_new_tokens):
        logits, hidden = model.decode(token, hidden, encoded, mask)
        # 예: 숫자 출력에는 한글·영어·특수 토큰이 섞이지 않도록 한다.
        logits = logits.masked_fill(~allowed, float("-inf"))
        token = logits.argmax(dim=-1)
        token_id = token.item()
        if token_id == EOS:
            break
        result.append(vocab[token_id])
    return " ".join(result) if target == "en" else "".join(result)


def evaluate(model, vocab, pair, rows):
    results = {}
    a, b = pair
    rows = rows_for_pair(rows, pair)
    for source, target in [(a, b), (b, a)]:
        if not rows:
            print(f"{source}->{target}: 평가할 공통 숫자가 없습니다.")
            results[f"{source}->{target}"] = {"accuracy": None, "count": 0, "by_length": {}, "errors": []}
            continue
        correct, errors, by_length = 0, [], {}
        for row in rows:
            prediction = translate(model, vocab, pair, row[source], source, target)
            matched = prediction == row[target]
            correct += matched
            digits = len(row["num"])
            counts = by_length.setdefault(digits, [0, 0])
            counts[0] += matched
            counts[1] += 1
            if not matched and len(errors) < 3:
                errors.append({"input": row[source], "answer": row[target], "prediction": prediction})
        accuracy = correct / len(rows)
        direction = f"{source}->{target}"
        results[direction] = {"accuracy": accuracy, "count": len(rows),
                              "by_length": by_length, "errors": errors}
        print(f"{direction}: {correct}/{len(rows)} = {accuracy:.2%}")
        print("자릿수별 [정답 수, 전체 수]:", dict(sorted(by_length.items())))
        for error in errors:
            print("  오답:", error)
    return results


## 8. 세 모델을 처음부터 학습하고 테스트
이 셀에서 실제 학습이 시작됩니다. 언어쌍별 데이터 수, epoch별 손실, 방향별 테스트 정확도를 출력합니다.
세 모델은 같은 숫자 분할을 사용하지만, 두 표현이 모두 있는 숫자 수에 따라 학습량이 달라질 수 있습니다.
공통 학습 숫자가 없는 언어쌍은 건너뛰며 다른 두 모델은 계속 학습합니다.
테스트가 없는 방향은 정확도를 계산하지 않습니다. `QUICK_RUN=True`의 수치는 실행 확인용입니다.

In [8]:
rows = load_rows(DATA_DIR)
train_rows, valid_rows, test_rows = split_rows(rows)
if QUICK_RUN:
    rng = random.Random(SEED)
    train_rows = rng.sample(train_rows, min(1000, len(train_rows)))
    valid_rows = rng.sample(valid_rows, min(100, len(valid_rows)))
    test_rows = rng.sample(test_rows, min(100, len(test_rows)))
vocab = make_vocab(train_rows)
stoi = {token: i for i, token in enumerate(vocab)}
print("전체 숫자:", len(rows), "| train/valid/test:", len(train_rows), len(valid_rows), len(test_rows))
print("사전 크기:", len(vocab))
trained_paths = {}
test_results = {}
for pair in PAIRS:
    print("\n", pair, "train/valid/test 숫자:",
          *(len(rows_for_pair(part, pair)) for part in (train_rows, valid_rows, test_rows)))
    path = train_pair(pair)
    if path is None:
        continue
    trained_paths[pair] = path
    model, saved_vocab, saved_pair = load_model(path)
    test_results.update(evaluate(model, saved_vocab, saved_pair, test_rows))
    print("모델 저장:", path.name)

print("\n이번 실행에서 직접 측정한 테스트 정확도")
for direction, metrics in test_results.items():
    accuracy = metrics["accuracy"]
    print(f"{direction}: 평가 자료 없음" if accuracy is None else
          f"{direction}: {accuracy:.4%} / 테스트 {metrics['count']:,}건")
total = sum(m["count"] for m in test_results.values())
correct = sum(round(m["accuracy"] * m["count"]) for m in test_results.values() if m["accuracy"] is not None)
if total:
    print(f"전체: {correct:,}/{total:,} = {correct / total:.4%}")
else:
    print("평가 가능한 테스트 숫자가 없습니다.")


데이터: 영어 100,000개 | 한글 100,000개 | 공통 숫자 100,000개
전체 숫자: 100000 | train/valid/test: 80001 9999 10000
사전 크기: 65

 ('num', 'en') train/valid/test 숫자: 80001 9999 10000
('num', 'en') 사용 숫자: 학습 80,001개 | 검증 9,999개
('num', 'en') epoch 1/20 | train 0.4784 | valid 0.0935 | 11.3초
('num', 'en') epoch 2/20 | train 0.0508 | valid 0.0294 | 9.1초
('num', 'en') epoch 3/20 | train 0.0173 | valid 0.0124 | 9.2초
('num', 'en') epoch 4/20 | train 0.0103 | valid 0.0092 | 8.4초
('num', 'en') epoch 5/20 | train 0.0074 | valid 0.0710 | 9.1초
('num', 'en') epoch 6/20 | train 0.0054 | valid 0.0044 | 9.1초
('num', 'en') epoch 7/20 | train 0.0043 | valid 0.0044 | 8.9초
('num', 'en') epoch 8/20 | train 0.0039 | valid 0.0029 | 8.5초
('num', 'en') epoch 9/20 | train 0.0032 | valid 0.0055 | 9.0초
('num', 'en') epoch 10/20 | train 0.0018 | valid 0.0020 | 9.1초
('num', 'en') epoch 11/20 | train 0.0035 | valid 0.0010 | 8.4초
('num', 'en') epoch 12/20 | train 0.0016 | valid 0.0014 | 9.3초
('num', 'en') epoch 13/20 | train 0.0036 | va

## 9. 이번에 학습한 모델 다시 읽고 예시 확인
현재 학습 셀에서 저장한 파일만 불러옵니다. 과거에 남은 다른 모델 파일은 사용하지 않습니다.
각 모델의 두 방향으로 예측합니다. 예시는 호출 방식 확인용이며 항상 정답이라는 뜻은 아닙니다.

In [9]:
models = {f"{a}_{b}": load_model(path) for (a, b), path in trained_paths.items()}
def translate_text(text, source, target):
    if source not in ("num", "en", "ko") or target not in ("num", "en", "ko"):
        raise ValueError("언어는 num, en, ko 중에서 선택하세요.")
    if source == target:
        raise ValueError("입력 언어와 출력 언어를 다르게 선택하세요.")
    for model, vocab, pair in models.values():
        if {source, target} == set(pair):
            return translate(model, vocab, pair, text, source, target)
    raise ValueError("해당 방향의 모델을 찾지 못했습니다.")

sample = {"num": "21546", "en": "twenty one thousand five hundred forty six",
          "ko": "이만 천오백사십육"}
for _, _, pair in models.values():
    for source, target in (pair, pair[::-1]):
        print(f"{source}->{target}: {translate_text(sample[source], source, target)}")
if not models:
    print("학습된 모델이 없습니다. 데이터와 학습 셀 출력을 확인하세요.")


num->en: twenty one thousand five hundred forty six
en->num: 21546
num->ko: 이만 천오백사십육
ko->num: 21546
en->ko: 이만 천오백사십육
ko->en: twenty one thousand five hundred forty six


## 10. 내 입력으로 번역
언어는 `num`(숫자), `en`(영어), `ko`(한글)입니다.
`TEXT`, `SOURCE`, `TARGET`을 바꾸고 이 셀만 다시 실행하면 재학습 없이 번역합니다.
앞에 불필요한 0이 붙은 숫자나 숫자 표현이 아닌 문장은 올바른 번역이 보장되지 않습니다.

In [10]:
TEXT = "14" #@param {type:"string"}
SOURCE = "num" #@param ["num", "en", "ko"]
TARGET = "en" #@param ["num", "en", "ko"]

try:
    result = translate_text(TEXT, SOURCE, TARGET)
except (ValueError, TypeError) as error:
    print("입력 오류:", error)
else:
    print("입력:", TEXT)
    print("결과:", result)


입력: 14
결과: fourteen


## 11. 이번 학습 모델 다운로드
이 셀은 이번 실행에서 저장한 `.pt`만 ZIP으로 묶어 다운로드합니다.
모든 언어쌍을 학습했다면 세 모델이 들어 있습니다. 학습하지 못한 언어쌍은 포함하지 않습니다.
`QUICK_RUN=True`라면 실행 확인용 모델입니다. 다운로드 파일명에도 `demo_`가 붙습니다.

In [11]:
if not trained_paths:
    raise ValueError("다운로드할 모델이 없습니다. 먼저 학습 셀을 실행하세요.")
model_zip = Path.cwd() / f"{PREFIX}project01_trained_models.zip"
with zipfile.ZipFile(model_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in trained_paths.values():
        archive.write(path, arcname=path.name)
print("다운로드 모델:", ", ".join(path.name for path in trained_paths.values()))
files.download(str(model_zip))


다운로드 모델: num_en.pt, num_ko.pt, en_ko.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. 마지막 셀: 저장된 파일을 로드하여 테스트 수행

**사용 방법**
1. 처음 실행할 때는 위 셀들을 순서대로 실행합니다. 학습 셀에서 저장한 `.pt`를 마지막 셀이 다시 읽습니다.
2. 학습이 끝난 같은 런타임에서는 **이 마지막 셀만 다시 실행**해도 됩니다. 가중치를 갱신하지 않습니다.
3. 학습 없이 저장 모델만 시험하려면 1~7절을 실행하여 설정·데이터 업로드·함수 정의를 준비하고,
   저장한 `.pt` 파일을 `MODEL_DIR` 폴더(기본 `/content`, GitHub에서 받은 모델 3개를 이 폴더에 업로드)에 둔 뒤 이 마지막 셀을 실행합니다.
   8절 학습 셀은 건너뜁니다. `QUICK_RUN`은 저장 모델을 만들 때와 같은 값으로 맞춥니다.
4. `EVAL_SAMPLES_PER_PAIR=100`은 언어쌍당 최대 100개 테스트 숫자를 평가합니다.
   **전체 테스트**는 이 값을 `0`으로 바꿔 실행합니다. 표에 평가 건수를 함께 표시합니다.

이 셀은 파일에서 모델을 새로 불러와 여섯 방향 예시의 정답과 예측을 비교한 뒤,
원본 데이터의 동일한 숫자 분할에서 테스트 부분만 골라 정확도를 직접 측정합니다.
예시의 PASS/FAIL과 테스트 정확도는 별개입니다. 데이터 파일이 없으면 예시 확인만 수행합니다.
JSON이나 과거에 저장한 정확도 표는 사용하지 않습니다.

In [12]:
# 사용 방법
# 1. 학습 후 이 마지막 셀만 다시 실행하면 저장된 .pt를 다시 읽어 테스트합니다. 재학습하지 않습니다.
# 2. 새 런타임에서는 1~7절을 실행하고, 저장 모델을 MODEL_DIR 폴더에 둔 뒤 이 셀을 실행합니다.
#    이 경우 8절 학습은 건너뛰고 QUICK_RUN을 모델 저장 당시 값으로 맞춥니다.
# 3. 아래 값은 언어쌍당 테스트 숫자 수입니다. 100은 일부 표본, 0은 전체 테스트입니다.
EVAL_SAMPLES_PER_PAIR = 100 #@param {type:"integer"}
if EVAL_SAMPLES_PER_PAIR < 0:
    raise ValueError("평가 개수는 0(전체) 또는 양수로 입력하세요.")

if "trained_paths" in globals():
    saved_paths = list(trained_paths.values())
else:
    saved_paths = [MODEL_DIR / f"{PREFIX}{a}_{b}.pt" for a, b in PAIRS]
reloaded_models = {}
for saved_path in saved_paths:
    if not saved_path.exists():
        print("건너뜀: 저장 모델이 없습니다:", saved_path)
        continue
    loaded_model, loaded_vocab, loaded_pair = load_model(saved_path)
    reloaded_models[loaded_pair] = (loaded_model, loaded_vocab, loaded_pair)
    print("저장 파일 로드 완료:", saved_path.name)
if not reloaded_models:
    raise ValueError("불러온 모델이 없습니다. 모델 폴더와 파일 이름을 확인하세요.")

examples = {"num": "21546", "en": "twenty one thousand five hundred forty six",
            "ko": "이만 천오백사십육"}
print("\n저장 모델의 방향별 예시 테스트 (전체 정확도와 별개)")
example_passed = 0
example_total = 0
for loaded_model, loaded_vocab, loaded_pair in reloaded_models.values():
    for source, target in (loaded_pair, loaded_pair[::-1]):
        prediction = translate(loaded_model, loaded_vocab, loaded_pair, examples[source], source, target)
        matched = prediction == examples[target]
        example_passed += matched
        example_total += 1
        print(f"{'PASS' if matched else 'FAIL'} | {source}->{target} | 입력: {examples[source]}")
        print(f"  정답: {examples[target]} | 예측: {prediction}")
print(f"예시 통과: {example_passed}/{example_total}")

last_test_results = {}
if all((DATA_DIR / name).exists() for name in ("num2txt.txt", "num2kor.txt")):
    _, _, saved_test_rows = split_rows(load_rows(DATA_DIR))
    print("\n저장 모델로 직접 계산한 테스트 정확도")
    for loaded_model, loaded_vocab, loaded_pair in reloaded_models.values():
        candidates = rows_for_pair(saved_test_rows, loaded_pair)
        if EVAL_SAMPLES_PER_PAIR:
            candidates = random.Random(SEED).sample(
                candidates, min(EVAL_SAMPLES_PER_PAIR, len(candidates)))
        last_test_results.update(evaluate(loaded_model, loaded_vocab, loaded_pair, candidates))
    evaluated_total = sum(m["count"] for m in last_test_results.values())
    evaluated_correct = sum(round(m["accuracy"] * m["count"])
                            for m in last_test_results.values() if m["accuracy"] is not None)
    if evaluated_total:
        label = "전체 테스트" if EVAL_SAMPLES_PER_PAIR == 0 else "테스트 일부 표본"
        print(f"{label} 합계: {evaluated_correct:,}/{evaluated_total:,} = {evaluated_correct / evaluated_total:.4%}")
else:
    print("원본 데이터가 없어 테스트 정확도 측정을 생략했습니다. 위 예시 결과를 확인하세요.")


저장 파일 로드 완료: num_en.pt
저장 파일 로드 완료: num_ko.pt
저장 파일 로드 완료: en_ko.pt

저장 모델의 방향별 예시 테스트 (전체 정확도와 별개)
PASS | num->en | 입력: 21546
  정답: twenty one thousand five hundred forty six | 예측: twenty one thousand five hundred forty six
PASS | en->num | 입력: twenty one thousand five hundred forty six
  정답: 21546 | 예측: 21546
PASS | num->ko | 입력: 21546
  정답: 이만 천오백사십육 | 예측: 이만 천오백사십육
PASS | ko->num | 입력: 이만 천오백사십육
  정답: 21546 | 예측: 21546
PASS | en->ko | 입력: twenty one thousand five hundred forty six
  정답: 이만 천오백사십육 | 예측: 이만 천오백사십육
PASS | ko->en | 입력: 이만 천오백사십육
  정답: twenty one thousand five hundred forty six | 예측: twenty one thousand five hundred forty six
예시 통과: 6/6
데이터: 영어 100,000개 | 한글 100,000개 | 공통 숫자 100,000개

저장 모델로 직접 계산한 테스트 정확도
num->en: 99/100 = 99.00%
자릿수별 [정답 수, 전체 수]: {4: [9, 9], 5: [17, 17], 6: [19, 19], 7: [17, 17], 8: [17, 17], 9: [6, 7], 10: [14, 14]}
  오답: {'input': '900031603', 'answer': 'nine hundred million thirty one thousand six hundred three', 'prediction': 'nine hundred millio